# 02 · Agricultural costs & inventory — form 50-сг (2021–2025)

Reproducible extraction from the **original** `data_pdf.zip`: row codes + **14 cost columns** and **11 inventory columns** using page geometry. It saves auditable numeric datasets, balance diagnostics, and a cautiously joined agricultural panel for later EDA/LP optimization.

**Before using the data for any decision:** review `agriculture_qa.csv` and `extraction_review.csv`, and manually compare representative PDF rows for **each year**. Dashes are treated as *missing/not reported* (`NaN`), **not automatically as zero**. The model never invents warehouse capacity, sale prices, or monthly observations.

**Sensitive data:** source PDFs and these extracts may identify a specific enterprise; keep them out of a public repo without explicit permission.

## 0. Requirements

Uses PyMuPDF, pandas, numpy. In Colab or a new environment, run the installation cell once. CI does not execute notebooks because raw reports are not committed.

In [1]:
%pip -q install "pymupdf>=1.24,<2" "pandas>=2.0,<3" "numpy>=1.26,<3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymupdf4llm 2.1 requires pymupdf==2.1, but you have pymupdf 1.28.2 which is incompatible.
pymupdf-layout 2.1 requires pymupdf==2.1, but you have pymupdf 1.28.2 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Locate input and output directories

Run from repo root or from `notebooks/`. Upload `data_pdf.zip` to `data/agriculture/raw/` (or current working directory).

In [2]:
from pathlib import Path
import zipfile, re, io, unicodedata, json
import pandas as pd
import numpy as np
import fitz
from IPython.display import display

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
choices = [ROOT/'data/agriculture/raw/data_pdf.zip',cwd/'data_pdf.zip',ROOT/'data_pdf.zip',Path('/content/data_pdf.zip')]
ZIP = next((p for p in choices if p.is_file()),None)
if ZIP is None: raise FileNotFoundError('Missing data_pdf.zip: put it in data/agriculture/raw/')
OUT = ROOT/'data/agriculture/processed'
OUT.mkdir(parents=True,exist_ok=True)
print('Input:',ZIP,'\nOutput:',OUT)

Input: /Users/margopaduchak/Desktop/прогнози/forecasting-and-decision-optimization/data/agriculture/raw/data_pdf.zip 
Output: /Users/margopaduchak/Desktop/прогнози/forecasting-and-decision-optimization/data/agriculture/processed


## 2. Source metadata and geometry templates

Templates are based on actual 50-сг pages in the supplied 2021–2025 archive (page numbering differs by year). If a different PDF rendering is supplied, strict parsing/QA checks should surface geometry mismatches. Changes in layout are **not** silently auto-corrected.

In [3]:
def norm(s):
    return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',str(s)).replace('\xa0',' ')).strip()

def classify(text,filename):
    if re.search(r'50\s*[-–]?\s*сг',text,re.I) or 'S27021' in filename: form='50-сг'
    elif re.search(r'29\s*[-–]?\s*сг',text,re.I) or 'S27029' in filename: form='29-сг'
    else: form=None
    match=re.search(r'за\s+(20\d{2})\s*(?:рік|року|р\.)',text,re.I)
    if not match: match=re.search(r'\b(202[1-5])\b',text[:3500])
    return form,int(match.group(1)) if match else None

# Left edges of cells in actual documents, oldest years have distinct printed grids.
ANCHORS = {
  2021:{'cost':[182,226,270,312,357,398,443,485,532,571,616,658,701,754],
        'inventory':[238,288,342,389,448,492,549,596,646,696,751]},
  2022:{'cost':[180,226,274,318,360,403,452,502,551,591,633,675,716,769],
        'inventory':[237,293,347,391,457,508,567,613,662,711,765]},
  2023:{'cost':[202,254,301,337,375,409,449,491,539,575,616,656,700,776],
        'inventory':[206,256,310,355,416,458,523,569,632,689,764]},
  2024:{'cost':[202,254,301,337,375,409,449,491,539,575,616,656,700,776],
        'inventory':[206,256,310,355,416,458,523,569,632,689,764]},
  2025:{'cost':[202,254,301,337,375,409,449,491,539,575,616,656,700,776],
        'inventory':[206,256,310,355,416,458,523,569,632,689,764]},
}
COST_NAMES=['production_cost_k_uah','direct_materials_k_uah','seed_k_uah','fertilizers_k_uah','fuel_k_uah','other_materials_k_uah','direct_labor_k_uah','other_direct_and_overhead_k_uah','social_contributions_k_uah','depreciation_k_uah','third_party_services_k_uah','other_expenses_k_uah','cogs_sold_k_uah','selling_expenses_k_uah']
INV_NAMES=['opening_stock_centners','inflow_centners','inflow_external_centners','outflow_centners','to_processors_centners','processed_centners','to_markets_centners','in_kind_payment_centners','feed_centners','seed_use_centners','storage_loss_centners']
assert len(COST_NAMES)==14 and len(INV_NAMES)==11

reports=[]
with zipfile.ZipFile(ZIP) as archive:
  for filename in archive.namelist():
    if not filename.lower().endswith('.pdf') or '__MACOSX' in filename: continue
    data=archive.read(filename)
    doc=fitz.open(stream=data,filetype='pdf')
    form,year=classify(doc[0].get_text(sort=True),filename)
    if form=='50-сг':reports.append({'file':Path(filename).name,'year':year,'bytes':data,'pages':len(doc)})
    doc.close()
print('50-сг reports:',[(r['year'],r['pages']) for r in reports])
assert len(reports)==5 and sorted(r['year'] for r in reports)==[2021,2022,2023,2024,2025]

50-сг reports: [(2023, 10), (2021, 8), (2024, 10), (2025, 10), (2022, 7)]


## 3. Extract coded table rows by column coordinates

Use **source row codes** (1101–1199 cost, 3001–3099 inventory), not the previous notebook's page-based candidate labels. Preserve every original cell in `raw_cells_json` and record the exact page for manual audit.

In [4]:
NUMBER_FRAGMENT=re.compile(r'^(?:\d[\d.,]*|[-–—хxХX])$')

def parse_num(text):
    s=norm(text)
    if s in ('','-','–','—','х','Х','x','X'): return np.nan
    s=s.replace(' ','').replace(',','.')
    try: return float(s)
    except ValueError:return np.nan

def pick_row(words,code_word,anchors):
    # Words on the same printed line, allowing small baseline differences.
    row=sorted([w for w in words if abs(w[1]-code_word[1])<3.3],key=lambda w:w[0])
    label=norm(' '.join(w[4] for w in row if w[2]<code_word[0]-2 and re.search('[А-Яа-яІіЇїЄєҐґ]',w[4])))
    boundaries=[(a+b)/2 for a,b in zip(anchors[:-1],anchors[1:])]
    cols=[[] for _ in anchors]
    for w in row:
        if w[0]<=code_word[2]+4 or not NUMBER_FRAGMENT.fullmatch(w[4]):continue
        x=w[0]
        ci=next((i for i,b in enumerate(boundaries) if x<b),len(anchors)-1)
        if x<anchors[0]-15 or x>anchors[-1]+35:continue
        cols[ci].append((w[0],w[4]))
    raw=[' '.join(t for _,t in sorted(group)) for group in cols]
    return label,raw

costs=[]; inventories=[]; warnings=[]
for report in reports:
    doc=fitz.open(stream=report['bytes'],filetype='pdf')
    yr=report['year']
    for page_index,p in enumerate(doc):
        words=p.get_text('words')
        for word in words:
            value=str(word[4])
            if not re.fullmatch(r'\d{4}',value):continue
            code=int(value)
            kind='cost' if 1101<=code<=1199 else 'inventory' if 3001<=code<=3099 else None
            if kind is None:continue
            # Reject row references in header footnotes, and plausible row-code positions only.
            if kind=='cost' and not (140<=word[0]<=185):continue
            if kind=='inventory' and not (155<=word[0]<=220):continue
            # Older PDF pages sometimes carry the section title on another page.
            # Limit to known section page ranges instead of requiring text on each page.
            if kind=='cost' and page_index+1 != 2: continue
            if kind=='inventory' and page_index+1 != (4 if yr<=2022 else 5): continue
            label,raw=pick_row(words,word,ANCHORS[yr][kind])
            if not label:
                # In early PDFs the crop label is printed on the line above the row code.
                # Assign an explicit code-based label instead of discarding the numeric row.
                fallbacks={'1103':'пшениця','3002':'пшениця','1104':'кукурудза на зерно','3003':'кукурудза на зерно'}
                label=fallbacks.get(value,'')
                if not label: continue
            names=COST_NAMES if kind=='cost' else INV_NAMES
            if not any(parse_num(t)==parse_num(t) for t in raw): continue # omit all-dash rows
            item={'year':yr,'row_code':value,'product_name_raw':label,'source_file':report['file'],
                  'source_page':page_index+1,'raw_cells_json':json.dumps(raw,ensure_ascii=False)}
            item.update({k:parse_num(v) for k,v in zip(names,raw)})
            (costs if kind=='cost' else inventories).append(item)
    doc.close()

cost=pd.DataFrame(costs).sort_values(['year','row_code']).reset_index(drop=True)
inv=pd.DataFrame(inventories).sort_values(['year','row_code']).reset_index(drop=True)
assert not cost.empty and not inv.empty,'No 50-сг measurements extracted; check report layout'
print('Cost rows:',len(cost),'Inventory rows:',len(inv))
display(pd.concat({'costs':cost.groupby('year').size(),'inventory':inv.groupby('year').size()},axis=1).fillna(0).astype(int))
display(cost[['year','row_code','product_name_raw','production_cost_k_uah']].head(15))
display(inv[['year','row_code','product_name_raw','opening_stock_centners','inflow_centners','outflow_centners']].head(15))

Cost rows: 38 Inventory rows: 52


,costs,inventory
year,,
2021,6,9
2022,7,9
2023,8,11
2024,9,12
2025,8,11


,year,row_code,product_name_raw,production_cost_k_uah
0,2021,1103,пшениця,39482.2
1,2021,1104,кукурудза на зерно,94549.9
2,2021,1105,ячмінь,19892.7
3,2021,1111,Боби сої,27136.3
4,2021,1113,Насіння ріпаку й кользу,29738.2
5,2021,1114,Насіння соняшнику,76160.2
6,2022,1102,зернобобові,113308.3
7,2022,1103,пшениця,37978.3
8,2022,1104,кукурудза на зерно,36273.0
9,2022,1105,ячмінь,39057.0


,year,row_code,product_name_raw,opening_stock_centners,inflow_centners,outflow_centners
0,2021,3002,пшениця,41469.0,87654.0,50290.0
1,2021,3003,кукурудза на зерно,117326.0,199280.0,121546.0
2,2021,3004,ячмінь,7177.0,38995.0,25665.0
3,2021,3011,Боби сої,22386.0,26189.0,22710.0
4,2021,3012,Насіння ріпаку й кользи,NaN,26512.0,26445.0
5,2021,3013,Насіння соняшнику,16728.0,69296.0,27412.0
6,2021,3020,Силос з культур кормових,160579.0,216650.0,197230.0
7,2021,3021,Сінаж з культур кормових,32012.0,110608.0,85913.0
8,2021,3022,Сіно,1813.0,5735.0,4885.0
9,2022,3002,пшениця,78833.0,72495.0,12718.0


## 4. Validation: arithmetic, duplicates, accounting relationships

Production cost (column 1) should equal direct materials (2) + labor (7) + other direct/overhead (8). Materials (2) generally sum categories (3–6), and other direct/overhead (8) generally sum (9–12). These are checked only when **all necessary values are reported**.

Inventory: ending balance is **derived** as opening + inflow − outflow (columns 1,2,4). Other outflow columns 5–11 describe subsets and **do not** replace total outflow, so they must never be subtracted again. An inferred year-end figure is *not* a reported physical inventory count.

In [5]:
qa=[]
def flag(kind,row,issue,detail,severity='warning'):
    qa.append({'severity':severity,'dataset':kind,'year':int(row['year']),'row_code':row['row_code'],
               'issue':issue,'detail':str(detail),'source_file':row['source_file'],'source_page':int(row['source_page'])})
for dataset,df in [('cost',cost),('inventory',inv)]:
    for _,r in df[df.duplicated(['year','row_code'],keep=False)].iterrows():flag(dataset,r,'duplicate year and row code','Check possible repeated table row')
    names=COST_NAMES if dataset=='cost' else INV_NAMES
    for _,r in df.iterrows():
        for name in names:
            if pd.notna(r[name]) and r[name]<0:flag(dataset,r,'negative reported value',name,'error')
        if not pd.notna(r[names[0]]):flag(dataset,r,'missing first numeric field',names[0])

for _,r in cost.iterrows():
    for target,parts,description in [
      ('production_cost_k_uah',['direct_materials_k_uah','direct_labor_k_uah','other_direct_and_overhead_k_uah'],'production cost sum'),
      ('direct_materials_k_uah',['seed_k_uah','fertilizers_k_uah','fuel_k_uah','other_materials_k_uah'],'materials sum'),
      ('other_direct_and_overhead_k_uah',['social_contributions_k_uah','depreciation_k_uah','third_party_services_k_uah','other_expenses_k_uah'],'other costs sum')]:
        if pd.notna(r[target]) and all(pd.notna(r[p]) for p in parts):
            delta=float(r[target]-sum(r[p] for p in parts))
            if abs(delta)>max(1.0,abs(r[target])*0.002):flag('cost',r,'cost arithmetic mismatch',f'{description}: delta={delta:.2f}')

inv['ending_stock_derived_centners']=inv['opening_stock_centners']+inv['inflow_centners']-inv['outflow_centners']
inv['ending_stock_derived_tonnes']=inv['ending_stock_derived_centners']/10
inv['opening_stock_tonnes']=inv['opening_stock_centners']/10
inv['inflow_tonnes']=inv['inflow_centners']/10
inv['outflow_tonnes']=inv['outflow_centners']/10
for _,r in inv.iterrows():
    if pd.notna(r.ending_stock_derived_centners) and r.ending_stock_derived_centners < -1:
        flag('inventory',r,'negative implied ending inventory',f'{r.ending_stock_derived_centners:.0f} centners','error')
    subset=['to_processors_centners','processed_centners','to_markets_centners','in_kind_payment_centners','feed_centners','seed_use_centners','storage_loss_centners']
    if pd.notna(r.outflow_centners) and all(pd.notna(r[s]) for s in subset):
        if sum(r[s] for s in subset)>r.outflow_centners+1: flag('inventory',r,'outflow subsets exceed total', 'Check row decoding')

# Compare previous derived ending against following reported opening (only for exact row code).
for code,g in inv.groupby('row_code'):
    g=g.sort_values('year')
    for i in range(1,len(g)):
        prev,cur=g.iloc[i-1],g.iloc[i]
        if cur.year==prev.year+1 and pd.notna(prev.ending_stock_derived_centners) and pd.notna(cur.opening_stock_centners):
            delta=float(cur.opening_stock_centners-prev.ending_stock_derived_centners)
            if abs(delta)>max(2,0.02*max(abs(cur.opening_stock_centners),abs(prev.ending_stock_derived_centners))):
                flag('inventory',cur,'year-to-year stock difference',f'opening minus prior derived closing = {delta:.1f} centners; may be corrections or decoding')

qa_df=pd.DataFrame(qa,columns=['severity','dataset','year','row_code','issue','detail','source_file','source_page'])
print('QA issue counts:',qa_df.groupby(['severity','issue']).size().to_dict() if len(qa_df) else 'none')
display(qa_df.head(30))

QA issue counts: {('warning', 'missing first numeric field'): 2}


,severity,dataset,year,row_code,issue,detail,source_file,source_page
0,warning,inventory,2021,3012,missing first numeric field,opening_stock_centners,610020003782954S270211610000002122021.pdf,4
1,warning,inventory,2024,3014,missing first numeric field,opening_stock_centners,S2702118_Γäû_50_╤ü╨│_╤Ç╤û╤ç╨╜╨░_╨ù╨▓╤û╤é_╨┐╤Ç╨...,5


## 5. Crosswalk: cost and inventory crop codes

Code 1103 (cost) corresponds to 3002 (inventory) for wheat, etc. We join by a **reviewable explicit crosswalk**, not substring similarity. This is a curated mapping for the supplied 50-сг versions; inspect any unresolved/new row codes.

In [6]:
# Form 50-сг column: wheat, maize, barley, rye, oats, buckwheat, millet, pulses, soybean, rapeseed, sunflower.
CROSSWALK={
 '1103':('3002','wheat'), '1104':('3003','maize'), '1105':('3004','barley'),
 '1106':('3005','rye'), '1107':('3006','oats'), '1108':('3007','buckwheat'),
 '1109':('3009','millet'), '1110':('3010','dry_pulses'), '1111':('3011','soybeans'),
 '1113':('3012','rapeseed'), '1114':('3013','sunflower')}
lookup=pd.DataFrame([{'cost_row_code':k,'inventory_row_code':v[0],'crop':v[1]} for k,v in CROSSWALK.items()])
# Cost data include aggregate rows: keep these in production_costs.csv, but exclude from crop-level joined panel.
crop_cost=cost.merge(lookup,left_on='row_code',right_on='cost_row_code',how='inner',validate='many_to_one')
crop_inv=inv.merge(lookup,left_on='row_code',right_on='inventory_row_code',how='inner',validate='many_to_one')
assert not crop_cost.duplicated(['year','crop']).any(),'Duplicate cost crop/year: inspect data'
assert not crop_inv.duplicated(['year','crop']).any(),'Duplicate inventory crop/year: inspect data'

cost_fields=['year','crop','production_cost_k_uah','direct_materials_k_uah','seed_k_uah','fertilizers_k_uah','fuel_k_uah','other_materials_k_uah','direct_labor_k_uah','other_direct_and_overhead_k_uah','cogs_sold_k_uah','selling_expenses_k_uah']
inv_fields=['year','crop','opening_stock_tonnes','inflow_tonnes','outflow_tonnes','ending_stock_derived_tonnes','storage_loss_centners']
panel=crop_cost[cost_fields].merge(crop_inv[inv_fields],on=['year','crop'],how='outer',validate='one_to_one',indicator=True)
panel['production_cost_uah']=panel['production_cost_k_uah']*1000
panel['cogs_sold_uah']=panel['cogs_sold_k_uah']*1000
panel['inflow_external_tonnes']=np.nan # keep unavailable if not joined, no manufactured values
panel.drop(columns='inflow_external_tonnes',inplace=True)
print('Joined rows:',len(panel),'coverage:',panel['_merge'].value_counts().to_dict())
display(panel.head(15))

Joined rows: 30 coverage: {'both': 30, 'left_only': 0, 'right_only': 0}


,year,crop,production_cost_k_uah,direct_materials_k_uah,seed_k_uah,fertilizers_k_uah,fuel_k_uah,other_materials_k_uah,direct_labor_k_uah,other_direct_and_overhead_k_uah,cogs_sold_k_uah,selling_expenses_k_uah,opening_stock_tonnes,inflow_tonnes,outflow_tonnes,ending_stock_derived_tonnes,storage_loss_centners,_merge,production_cost_uah,cogs_sold_uah
0,2021,barley,19892.7,10229.0,1445.5,3060.8,2337.8,3384.9,1983.6,7680.1,10014.4,NaN,717.7,3899.5,2566.5,2050.7,NaN,both,19892700.0,10014400.0
1,2021,maize,94549.9,47919.4,6692.8,12025.5,5695.1,23506.0,4408.0,42222.5,36602.4,NaN,11732.6,19928.0,12154.6,19506.0,NaN,both,94549900.0,36602400.0
2,2021,rapeseed,29738.2,14537.4,1286.0,4284.1,2272.9,6694.4,1928.5,13272.3,27397.0,NaN,NaN,2651.2,2644.5,NaN,NaN,both,29738200.0,27397000.0
3,2021,soybeans,27136.3,13852.8,879.7,5061.3,2435.2,5476.6,2066.3,11217.2,20535.0,NaN,2238.6,2618.9,2271.0,2586.5,NaN,both,27136300.0,20535000.0
4,2021,sunflower,76160.2,32660.6,3404.5,8934.3,5682.1,14639.7,4821.3,38678.3,26442.4,NaN,1672.8,6929.6,2741.2,5861.2,NaN,both,76160200.0,26442400.0
5,2021,wheat,39482.2,22023.8,1590.2,8740.0,3571.6,8122.0,3030.5,14427.9,16352.0,NaN,4146.9,8765.4,5029.0,7883.3,NaN,both,39482200.0,16352000.0
6,2022,barley,39057.0,19694.4,3048.4,6833.4,4871.4,4941.2,2871.3,16491.3,42928.5,NaN,2050.7,7127.1,8755.2,422.6,NaN,both,39057000.0,42928500.0
7,2022,maize,36273.0,22951.4,3575.9,6206.0,4100.0,9069.5,1827.2,11494.4,55938.5,NaN,19506.0,6256.2,15724.1,10038.1,NaN,both,36273000.0,55938500.0
8,2022,rapeseed,56169.2,29806.6,2099.9,12106.4,4871.4,10728.9,2871.3,23491.3,50386.6,NaN,6.7,4407.8,4413.8,0.7,NaN,both,56169200.0,50386600.0
9,2022,soybeans,37967.3,23259.3,3206.3,8878.7,4650.0,6524.3,2740.8,11967.2,508.0,NaN,2586.5,3449.0,973.9,5061.6,NaN,both,37967300.0,508000.0


## 6. Export validated datasets + manual review sample

`production_costs.csv` and `inventory_flows.csv` include source provenance and all numeric columns. `agriculture_panel.csv` is the explicit crop-level join. `extraction_review.csv` contains example records selected by year and dataset for source-PDF comparison.

In [7]:
cost.to_csv(OUT/'production_costs.csv',index=False,encoding='utf-8-sig')
inv.to_csv(OUT/'inventory_flows.csv',index=False,encoding='utf-8-sig')
panel.to_csv(OUT/'agriculture_panel.csv',index=False,encoding='utf-8-sig')
qa_df.to_csv(OUT/'agriculture_qa.csv',index=False,encoding='utf-8-sig')
review=[]
for kind,df,first in [('cost',cost,'production_cost_k_uah'),('inventory',inv,'opening_stock_centners')]:
  for year,g in df.groupby('year'):
    preferred=g[g.row_code.isin(['1103','1104','1111'] if kind=='cost' else ['3002','3003','3011'])]
    s=preferred if not preferred.empty else g
    for _,r in s.head(3).iterrows():
        review.append({'dataset':kind,'year':year,'row_code':r.row_code,'label':r.product_name_raw,
            'first_numeric_value':r[first],'raw_cells_json':r.raw_cells_json,
            'source_file':r.source_file,'source_page':r.source_page,'manually_checked':False})
review_df=pd.DataFrame(review)
review_df.to_csv(OUT/'extraction_review.csv',index=False,encoding='utf-8-sig')
for name in ['production_costs','inventory_flows','agriculture_panel','agriculture_qa','extraction_review']:
  print(f'{name}.csv: {(OUT/(name+".csv")).stat().st_size:,} bytes')
print('Manual review required:',len(review_df),'sample rows')
print('NEXT: verify PDF samples for all 5 years; fix mismatches before optimization.')

production_costs.csv: 20,263 bytes
inventory_flows.csv: 22,907 bytes
agriculture_panel.csv: 4,450 bytes
agriculture_qa.csv: 571 bytes
extraction_review.csv: 11,832 bytes
Manual review required: 30 sample rows
NEXT: verify PDF samples for all 5 years; fix mismatches before optimization.
